# Arabidopsis Gene Expression Analysis Under Abiotic Stress

This notebook provides an interactive exploration of *Arabidopsis thaliana* gene expression data under four conditions: **Control**, **Drought**, **Heat**, and **Cold**.

---

### Biological Context

Plants are constantly exposed to environmental stresses that limit their growth and productivity. Understanding how plants respond to stresses like drought, heat, and cold at the molecular level is crucial for developing stress-tolerant crops.

### Analysis Pipeline
1. **Data loading** - Load simulated expression data
2. **Quality control** - Filter lowly-expressed genes, check for outliers
3. **Normalization** - Quantile normalization
4. **PCA** - Dimensionality reduction to visualize sample relationships
5. **Differential Expression** - Identify genes responding to each stress
6. **GO Enrichment** - Biological processes enriched in stress responses
7. **Visualization** - Publication-quality figures

In [ ]:
# Import libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import project modules
from src.data_generation import generate_dataset
from src.preprocessing import run_preprocessing
from src.analysis import run_full_analysis, run_pca, differential_expression, go_enrichment
from src.visualization import (
    plot_pca, plot_volcano, plot_heatmap,
    plot_marker_expression, plot_go_enrichment,
    CONDITION_COLORS
)

# Set up plotting
%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})
sns.set_style('whitegrid')
print('All imports successful!')

---
## 1. Generate or Load Data

We use simulated *Arabidopsis thaliana* gene expression data. The simulation is designed to produce realistic patterns:
- **5000 genes** with basal expression levels
- **4 biological replicates** per condition (Control, Drought, Heat, Cold)
- **Known stress markers** (e.g., RD29A for drought, HSP70 for heat, CBF3 for cold) have realistic fold-changes
- Technical noise follows a realistic distribution

In [ ]:
# Generate the dataset
expression_df, metadata_df = generate_dataset(
    n_genes=5000,
    samples_per_condition=4,
    seed=42,
)

print(f"Expression matrix: {expression_df.shape[0]} genes, {expression_df.shape[1]} samples")
print(f"Conditions: {metadata_df['Condition'].unique()}")
print(f"\nFirst few rows of expression data:")
display(expression_df.head())

In [ ]:
# Quick look at the data distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of expression values
expression_unstacked = expression_df.unstack().reset_index()
expression_unstacked.columns = ['Sample', 'Gene', 'Expression']
expression_unstacked['Condition'] = expression_unstacked['Sample'].map(
    metadata_df['Condition'].to_dict()
)

sns.boxplot(data=expression_unstacked, x='Condition', y='Expression',
            palette=CONDITION_COLORS, ax=axes[0],
            order=['Control', 'Drought', 'Heat', 'Cold'])
axes[0].set_title('Expression Distribution by Condition')
axes[0].set_ylabel('Expression (log₂ TPM+1)')

# Mean-variance relationship
means = expression_df.mean(axis=1)
vars_ = expression_df.var(axis=1)
axes[1].scatter(means, vars_, alpha=0.3, s=3, c='#2E86AB')
axes[1].set_xlabel('Mean Expression')
axes[1].set_ylabel('Variance')
axes[1].set_title('Mean-Variance Relationship')
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

---
## 2. Preprocessing

- **Step 1**: Filter out lowly-expressed genes (mean < 1 in > 50% of samples)
- **Step 2**: Quantile normalization to make samples comparable
- **Step 3**: Outlier detection using robust Z-scores

In [ ]:
preprocessed = run_preprocessing(
    expression_df, metadata_df,
    min_mean=1.0,
    min_samples_pct=0.5,
    normalize=True,
)

expr_norm = preprocessed['expression']
meta = preprocessed['metadata']
print(f"\nAfter preprocessing: {expr_norm.shape[0]} genes, {expr_norm.shape[1]} samples")

---
## 3. Principal Component Analysis (PCA)

PCA helps us visualize the high-dimensional expression data in 2D. If the stress treatments have strong effects on gene expression, we expect to see clear separation between conditions.

In [ ]:
# Run PCA
pca_results = run_pca(expr_norm)

# Create the PCA plot ourselves for interactive display
scores = pca_results['scores']
variance = pca_results['variance']

plot_df = scores.copy()
plot_df['Condition'] = meta.loc[plot_df.index.intersection(meta.index), 'Condition']

var_pc1 = variance.loc[variance['PC'] == 'PC1', 'Explained_Variance'].values[0]
var_pc2 = variance.loc[variance['PC'] == 'PC2', 'Explained_Variance'].values[0]

fig, ax = plt.subplots(figsize=(8, 6))
for condition in CONDITION_COLORS:
    subset = plot_df[plot_df['Condition'] == condition]
    ax.scatter(subset['PC1'], subset['PC2'],
               c=CONDITION_COLORS[condition], label=condition,
               s=150, edgecolors='white', linewidth=1, alpha=0.9, zorder=3)
    for idx, row in subset.iterrows():
        ax.annotate(idx, (row['PC1'], row['PC2']),
                    fontsize=8, ha='center', va='bottom',
                    xytext=(0, 8), textcoords='offset points')

ax.set_xlabel(f'PC1 ({var_pc1:.1%} variance)', fontweight='bold')
ax.set_ylabel(f'PC2 ({var_pc2:.1%} variance)', fontweight='bold')
ax.set_title('PCA: Sample Clustering by Condition', fontweight='bold', fontsize=14)
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.axhline(y=0, color='grey', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='grey', linestyle='--', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

# Explained variance table
print("\nExplained Variance by Principal Component:")
display(variance)

---
## 4. Differential Expression Analysis

We identify genes that significantly change expression under each stress condition compared to the Control.

**Method**: Welch's t-test with Benjamini-Hochberg FDR correction
- Threshold: |log₂FC| ≥ 1.0, adjusted p-value < 0.05

In [ ]:
# Drought vs Control
de_drought = differential_expression(
    expr_norm, 'Control', 'Drought', meta,
    log2fc_threshold=1.0, pval_threshold=0.05,
)
print(f"\nTop 10 DE genes (Drought vs Control):")
display(de_drought[['Gene', 'Log2FC', 'Adj_P_Value', 'Regulation']].head(10))

In [ ]:
# Heat vs Control
de_heat = differential_expression(
    expr_norm, 'Control', 'Heat', meta,
    log2fc_threshold=1.0, pval_threshold=0.05,
)
print(f"\nTop 10 DE genes (Heat vs Control):")
display(de_heat[['Gene', 'Log2FC', 'Adj_P_Value', 'Regulation']].head(10))

In [ ]:
# Cold vs Control
de_cold = differential_expression(
    expr_norm, 'Control', 'Cold', meta,
    log2fc_threshold=1.0, pval_threshold=0.05,
)
print(f"\nTop 10 DE genes (Cold vs Control):")
display(de_cold[['Gene', 'Log2FC', 'Adj_P_Value', 'Regulation']].head(10))

---
## 5. Volcano Plots

Volcano plots show the relationship between fold-change and statistical significance. Genes in the top-left are significantly downregulated, and genes in the top-right are significantly upregulated.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

de_dict = {
    'Drought vs Control': de_drought,
    'Heat vs Control': de_heat,
    'Cold vs Control': de_cold,
}

for ax, (title, df) in zip(axes, de_dict.items()):
    df = df.copy()
    df['-log10(p_adj)'] = -np.log10(df['Adj_P_Value'].clip(lower=1e-300))
    
    colors_v = {'Up': '#B2182B', 'Down': '#2166AC', 'NS': '#888888'}
    for reg_status in ['Up', 'Down', 'NS']:
        subset = df[df['Regulation'] == reg_status]
        if len(subset) > 0:
            ax.scatter(subset['Log2FC'], subset['-log10(p_adj)'],
                      c=colors_v[reg_status], label=f'{reg_status}' if reg_status != 'NS' else 'NS',
                      s=8, alpha=0.6 if reg_status == 'NS' else 0.9, rasterized=True)
    
    ax.axhline(-np.log10(0.05), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axvline(1.0, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axvline(-1.0, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
    
    # Label top 5 genes
    top = df[df['Significant']].head(5)
    for _, row in top.iterrows():
        ax.annotate(row['Gene'], (row['Log2FC'], row['-log10(p_adj)']),
                   fontsize=7, ha='center', va='bottom',
                   xytext=(0, 5), textcoords='offset points',
                   arrowprops=dict(arrowstyle='-', color='grey', alpha=0.3))
    
    ax.set_xlabel('Log₂ Fold Change', fontweight='bold')
    ax.set_ylabel('-Log₁₀ Adj. P-value', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8, markerscale=3)
    ax.grid(True, alpha=0.15)
    sns.despine()

plt.tight_layout()
plt.show()

---
## 6. Marker Gene Expression

Let's look at how known stress-marker genes behave across conditions. This validates that our simulated data produces biologically meaningful patterns.

In [ ]:
# Selected marker genes with known roles in stress response
marker_genes = {
    'RD29A': 'Drought-responsive (desiccation protectant)',
    'HSP70': 'Heat shock protein (protein folding chaperone)',
    'CBF3': 'Cold-binding factor (cold acclimation master regulator)',
    'NCED3': 'ABA biosynthesis (drought signaling)',
    'COR15A': 'Cold-regulated (membrane stabilizer)',
    'RBCS1A': 'Rubisco small subunit (photosynthesis, downregulated under stress)',
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, (gene, description) in enumerate(marker_genes.items()):
    if gene not in expr_norm.index:
        continue
    ax = axes[i]
    gene_expr = expr_norm.loc[gene]
    plot_data = gene_expr.to_frame('Expression')
    plot_data['Condition'] = meta.loc[plot_data.index, 'Condition']
    
    # Order conditions
    condition_order = ['Control', 'Drought', 'Heat', 'Cold']
    condition_order = [c for c in condition_order if c in plot_data['Condition'].values]
    
    sns.boxplot(data=plot_data, x='Condition', y='Expression',
               palette=CONDITION_COLORS, ax=ax, width=0.5, order=condition_order)
    sns.stripplot(data=plot_data, x='Condition', y='Expression',
                 color='black', ax=ax, size=7, edgecolor='white',
                 linewidth=0.5, jitter=True, order=condition_order, alpha=0.8)
    
    ax.set_title(f'{gene}\n{description}', fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Expression (log₂ TPM+1)')
    ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.suptitle('Expression Patterns of Known Stress Marker Genes',
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

---
## 7. GO Enrichment Analysis

Gene Ontology (GO) enrichment identifies biological processes that are overrepresented among the differentially expressed genes.

In [ ]:
# Run GO enrichment for each comparison
go_drought = go_enrichment(de_drought, expr_norm.index.tolist())
go_heat = go_enrichment(de_heat, expr_norm.index.tolist())
go_cold = go_enrichment(de_cold, expr_norm.index.tolist())

print("\n=== GO Enrichment: Drought ===")
if not go_drought.empty:
    display(go_drought[['GO_Term', 'DE_Genes_in_Term', 'Adj_P_Value', 'Significant']].head(10))

print("\n=== GO Enrichment: Heat ===")
if not go_heat.empty:
    display(go_heat[['GO_Term', 'DE_Genes_in_Term', 'Adj_P_Value', 'Significant']].head(10))

print("\n=== GO Enrichment: Cold ===")
if not go_cold.empty:
    display(go_cold[['GO_Term', 'DE_Genes_in_Term', 'Adj_P_Value', 'Significant']].head(10))

---
## 8. Summary & Interpretation

### Key Findings:

1. **PCA separates conditions clearly**: The first two principal components capture the major sources of variation, with stress-treated samples separating from controls along PC1, and individual stress types separating along PC2.

2. **Stress-specific responses**: Each stress condition triggers both shared and unique transcriptional programs:
   - **Drought**: Upregulation of ABA-responsive genes (RD29A, NCED3, LEA14)
   - **Heat**: Induction of heat shock proteins (HSP70, HSP90.1, HSP101)
   - **Cold**: Activation of CBF regulon (CBF1-3, COR15A, COR47)

3. **Photosynthesis downregulation**: Genes involved in photosynthesis (RBCS, LHCB) are consistently downregulated across all stress conditions, reflecting the growth-defense tradeoff.

4. **GO enrichment**: Biological processes related to stress response, ABA signaling, and temperature sensing are enriched among DE genes, confirming the biological relevance of the analysis.

### Biological Significance:

Understanding how plants respond to abiotic stress at the transcriptome level is essential for:
- **Crop improvement**: Identifying key regulatory genes for breeding stress-tolerant varieties
- **Climate adaptation**: Predicting how plants will respond to changing environmental conditions
- **Biotechnology**: Engineering stress-resistant crops through targeted gene modification